# UCR Archive EDA Notebook
General notebook to explore UCR datasets (not tied to one dataset).

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Resolve the UCR archive root across common folder layouts
_root_candidates = [
    (Path('..') / '..' / 'data' / 'UCRArchive_2018').resolve(),
    (Path('..') / '..' / 'data' / 'UCRArchive_2018' / 'UCRArchive_2018').resolve(),
]
UCR_ROOT = next((p for p in _root_candidates if p.exists()), _root_candidates[0])
UCR_ROOT


WindowsPath('D:/repositories/personal/xai-spatio-temporal/data/UCRArchive_2018/UCRArchive_2018')

In [ ]:
# Build a summary table for all datasets with TRAIN/TEST pairs
rows = []
for train_path in sorted(UCR_ROOT.rglob('*_TRAIN.tsv')):
    test_path = train_path.with_name(train_path.name.replace('_TRAIN.tsv', '_TEST.tsv'))
    if not test_path.exists():
        continue

    dataset = train_path.stem.replace('_TRAIN', '')
    train_df = pd.read_csv(train_path, sep='\t', header=None)
    test_df = pd.read_csv(test_path, sep='\t', header=None)

    rows.append({
        'dataset': dataset,
        'train_rows': len(train_df),
        'test_rows': len(test_df),
        'total_rows': len(train_df) + len(test_df),
        'columns': train_df.shape[1],
        'features': train_df.shape[1] - 1
    })

if not rows:
    raise ValueError(
        f'No *_TRAIN.tsv/*_TEST.tsv dataset pairs found under: {UCR_ROOT}. '
        'Check UCR_ROOT and folder structure.'
    )

summary_df = pd.DataFrame(rows).sort_values('total_rows', ascending=False).reset_index(drop=True)
summary_df.head(15)


KeyError: 'total_rows'

In [ ]:
# Choose any dataset here
dataset_name = 'ECG200'
train_path = UCR_ROOT / dataset_name / f'{dataset_name}_TRAIN.tsv'
test_path = UCR_ROOT / dataset_name / f'{dataset_name}_TEST.tsv'

train_df = pd.read_csv(train_path, sep='\t', header=None)
test_df = pd.read_csv(test_path, sep='\t', header=None)

print('Dataset:', dataset_name)
print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
print('Feature count:', train_df.shape[1] - 1)

display(train_df.head())
print('Class distribution (train):')
print(train_df[0].value_counts().sort_index())

In [ ]:
# Plot a few sample time series from selected dataset
num_samples = 5
sample_df = train_df.sample(min(num_samples, len(train_df)), random_state=42)

plt.figure(figsize=(10, 5))
for _, row in sample_df.iterrows():
    label = int(row.iloc[0])
    series = row.iloc[1:].to_numpy()
    plt.plot(series, alpha=0.8, linewidth=1.2, label=f'class {label}')

plt.title(f'Sample Time Series: {dataset_name}')
plt.xlabel('Time step')
plt.ylabel('Value')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()